# gedi_tiler_aggregate quickstart

GEDI-H3's `gh3_aggregate`, `gh3_rasterize` and `gh3_from_img` on the tiled GEDI database.
Each tool is also a shell command: `gt_aggregate("...")` here is `gt_aggregate ...` in a terminal.

Use the `gtagg` conda environment (see the README).

In [ ]:
from gtagg.cli import gt_aggregate, gt_from_img, gt_rasterize

# Where to write datasets: on S3 for DPS jobs, or a local directory for local runs.
OUT = "s3://maap-ops-workspace/YOUR_USERNAME/gtagg_quickstart"

## 1. Aggregate

Mean and 95th percentile RH98 on the 1 km EGI grid (the GEDI L4B grid), quality filtered,
over a small box. This runs here; add `--dps N` to run it as N DPS jobs instead.

Variables are database columns or `name=SQL`; SQL lists are 1-indexed, so `rh_l2a[99]` is RH98.

In [ ]:
gt_aggregate(f"-o {OUT}/rh98_egi6 -egi 6 -l rh98=rh_l2a[99] -a \"['mean','p95']\" -y -r=-85,44,-84,45")

The dataset is a directory of GeoParquet files, one per partition, readable with DuckDB or geopandas:

In [ ]:
from gtiler.database import ducky
con = ducky.init_duckdb()
con.sql(f"SELECT * FROM '{OUT}/rh98_egi6/parts/*.parquet' LIMIT 5")

## 2. Rasterize

In [ ]:
gt_rasterize(f"{OUT}/rh98_egi6 -o rasters/rh98_egi6 --merge")

import rasterio, matplotlib.pyplot as plt
with rasterio.open("rasters/rh98_egi6/merged.tif") as r:
    plt.imshow(r.read(1), cmap="viridis"); plt.colorbar(label=r.descriptions[0])

## 3. H3 hexagons, as a yearly time series

Column-specific aggregations go in a dict. Each time window gets its own raster.

In [ ]:
gt_aggregate(f"""-o {OUT}/h3_6_yearly -h3 6 -l rh98=rh_l2a[99] agbd_l4a
         -a "{{'rh98': ['mean', 'count'], 'agbd_l4a': 'median'}}" -y -r=-85,44,-84,45
         -t0 2020-01-01 -t1 2023-01-01 -ti 1 -tu years""")
gt_rasterize(f"{OUT}/h3_6_yearly -o rasters/h3_6_yearly --merge")

## 4. Sample a raster at every shot

`-B` selects bands (0-based), `-b` names them, and `-w BZO` adds window statistics:
band B, odd window size Z, operation O (0 sum, 1 mean, 2 median, 3 mode, 4 std, 5 min, 6 max,
7 count, 8 range). The region defaults to the raster's extent.

In [ ]:
IMAGE = "s3://path/to/your/raster.tif"  # any GDAL raster or VRT, in any CRS
gt_from_img(f"-o {OUT}/sampled -i {IMAGE} -B 0 -b value -w 031 033 -y -g")
con.sql(f"SELECT * FROM '{OUT}/sampled/parts/*.parquet' LIMIT 5")

## 5. Large regions on DPS

Each job works through its share of partitions (~160 km EGI squares or H3 level 3 cells) one at a time,
so memory doesn't depend on the number of jobs; choose it for wall time.

In [ ]:
DS = f"{OUT}/rh98_conus_egi6"
gt_aggregate(f"-o {DS} -egi 6 -l rh98=rh_l2a[99] -a mean -y -r conus.gpkg --dps 20")

Follow the batch at the console link printed above. When it ends, run the same cell again:
it resubmits any partitions that failed, as the batch's next generation, or says that all are finished.

In [ ]:
gt_rasterize(f"{DS} -o rasters/rh98_conus_egi6 --merge")